In [ ]:
python src/load_to_sql.py

In [6]:
from pathlib import Path
print("cwd:", Path.cwd())
print("db folder exists:", (root / "data" / "processed").exists())
print("root used:", root)

cwd: /Users/abhisheksharma/Documents/support-ticket-analytics/sql
db folder exists: False
root used: /Users/abhisheksharma/Documents/support-ticket-analytics/sql


In [2]:
import sqlite3
from pathlib import Path
import pandas as pd

root = Path.cwd()
while not (root / "data").exists() and root != root.parent:
    root = root.parent

DB_PATH = root / "data" / "processed" / "tickets.db"
print("Using:", DB_PATH, "| exists:", DB_PATH.exists())

conn = sqlite3.connect(DB_PATH)

def run(query):
    return pd.read_sql_query(query, conn)

run("SELECT COUNT(*) AS total_tickets FROM tickets")

Using: /Users/abhisheksharma/Documents/support-ticket-analytics/data/processed/tickets.db | exists: True


,total_tickets
0,8469


In [3]:
run("""
SELECT ticket_status,
       COUNT(*) AS tickets,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM tickets), 1) AS pct
FROM tickets
GROUP BY ticket_status
ORDER BY tickets DESC
""")

,ticket_status,tickets,pct
0,Pending Customer Response,2881,34.0
1,Open,2819,33.3
2,Closed,2769,32.7


In [4]:
run("""
SELECT ticket_priority,
       COUNT(*) AS total,
       SUM(is_closed) AS closed,
       ROUND(100.0 * SUM(is_closed) / COUNT(*), 1) AS pct_closed
FROM tickets
GROUP BY ticket_priority
ORDER BY pct_closed DESC
""")

,ticket_priority,total,closed,pct_closed
0,Critical,2129,726,34.1
1,High,2085,705,33.8
2,Medium,2192,694,31.7
3,Low,2063,644,31.2


In [5]:
run("""
SELECT ticket_type,
       COUNT(*) AS rated_tickets,
       ROUND(AVG(customer_satisfaction_rating), 2) AS avg_rating
FROM tickets
WHERE is_closed = 1
GROUP BY ticket_type
ORDER BY avg_rating DESC
""")

,ticket_type,rated_tickets,avg_rating
0,Cancellation request,516,3.03
1,Billing inquiry,544,3.03
2,Product inquiry,533,3.02
3,Technical issue,580,2.96
4,Refund request,596,2.93


In [6]:
run("SELECT MIN(customer_satisfaction_rating) AS lowest, MAX(customer_satisfaction_rating) AS highest FROM tickets")

,lowest,highest
0,1.0,5.0


In [7]:
out = root / "reports"
out.mkdir(exist_ok=True)

run("SELECT * FROM tickets").to_csv(out / "tickets_for_dashboard.csv", index=False)
print("Saved:", out / "tickets_for_dashboard.csv")

Saved: /Users/abhisheksharma/Documents/support-ticket-analytics/reports/tickets_for_dashboard.csv
